# 01 · Hyperparameter tuning — grid search per unit and per family

## Executive summary (read this first)

**The model** is the baseline: the **cumulative random walk + Cholesky + EWMA** (with optional Student-t shocks). It has
four hyperparameters. The grid tries every combination, **480 in all**:

| hyperparameter | what it does | grid |
|---|---|---|
| `window` | past steps for the correlation between assets (and for the sd when EWMA is off) | 42, 130, 260, 520 |
| `halflife` | EWMA sd: a step's weight halves every `halflife` steps; `off` = plain sd of `window` | 21, 42, 63, 126, 252, off |
| `nu` (ν) | Student-t shocks, smaller = fatter tails; `normal` = no t | 4, 5, 8, normal |
| `widen` | multiplies the sd | 0.9, 1.0, 1.1, 1.2, 1.3 |

**The data.** Each unit's own history, split by time: **60 % train, 20 % validation, 20 % test**. A "date" is a past
forecast origin; its answer is the value `horizon` steps later. Where a date's answer would land in the next block, that
date is dropped (the purge), so no answer leaks from one block into the next.

**The procedure.**

1. Score **all 480 settings on validation**, every unit.
2. **Best per unit** = the setting with the lowest validation score on that unit.
   **Global per family** = the one setting with the lowest validation score across the family's units (geometric
   mean, so every unit counts the same whatever its scale).
3. Score the picks on **train** and **test** too. Test is looked at once, after the picks are fixed.

**The score** is each unit's own **total score** from its `card.toml`: `0.5 × CRPS + 0.3 × joint + 0.2 × tail` (on a
one-cell unit the joint weight is spread over the other two). Lower is better. Raw, in the unit's own units; only the
total is reported. The validation score of a pick is optimistic (it was picked on validation); **test is the honest
number.**

Results: §6 (per unit), §7 (per family), §9 (in short). No realized outcome is read: every answer is a later row of the
unit's own history.

## 0 — Setup

In [1]:
import json, multiprocessing as mp, pathlib, subprocess, sys, tempfile, time, tomllib
import numpy as np, pandas as pd

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "hyperparameter_tuning" else pathlib.Path.cwd() / "hyperparameter_tuning"
REPO, UNITS, PREP, OUT = HERE.parent, HERE.parent / "units", HERE.parent / "model_baseline" / "data", HERE / "data"
OUT.mkdir(exist_ok=True)
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
missing = [f for f in ("targets.parquet", "steps.parquet") if not (PREP / f).exists()]
assert not missing, f"run model_baseline/01_prep_data.ipynb first: it writes {missing} to model_baseline/data/"
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")

TARGETS = pd.read_parquet(PREP / "targets.parquet")      # every past origin of every unit, with its answer
STEPS = pd.read_parquet(PREP / "steps.parquet")          # every unit's daily (or monthly) steps
CARDS = {u: tomllib.loads((UNITS / u / "card.toml").read_text()) for u in sorted(TARGETS.unit.unique())}
FAMILY = {u: CARDS[u]["metadata"]["category"] for u in CARDS}
NASSETS = {u: len(CARDS[u]["targets"]["asset_ids"]) for u in CARDS}
FAMS = sorted(set(FAMILY.values()))
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))     # geometric mean
print(f"{len(CARDS)} units: " + ", ".join(f"{f} {sum(v == f for v in FAMILY.values())}" for f in FAMS) + f"; {WORKERS} workers")

103 units: T2-F1 23, T2-F2 27, T2-F3 22, T2-F4 31; 10 workers


## 1 — The split: 60 % train, 20 % validation, 20 % test

For each unit: all its past origins, oldest first; the first 60 % are train, the next 20 % validation, the last 20 %
test. Then the purge: a train date whose answer falls on or after the first validation date is dropped, and the same
for validation against test.

In [2]:
BLOCKS3 = ["train", "validation", "test"]

def split_unit(u):
    """origin date -> 'train' / 'validation' / 'test' / 'purged'."""
    t = TARGETS[(TARGETS.unit == u) & (TARGETS.split != "predict")]
    answer = t.groupby("origin_date").target_date.max().sort_index()   # the last answer date of each origin
    dates, n = answer.index.to_numpy(), len(answer)
    cut = [int(round(0.6 * n)), int(round(0.8 * n))]
    block = np.array(["train"] * cut[0] + ["validation"] * (cut[1] - cut[0]) + ["test"] * (n - cut[1]), dtype=object)
    for c, name in zip(cut, ["train", "validation"]):
        block[(block == name) & (answer.to_numpy() >= dates[c])] = "purged"
    return pd.Series(block, index=pd.DatetimeIndex(dates), name="block")

SPLIT = {u: split_unit(u) for u in CARDS}
rows = []
for u, s in SPLIT.items():
    for b in [*BLOCKS3, "purged"]:
        d = s[s == b].index
        rows.append({"unit": u, "family": FAMILY[u], "block": b, "dates": len(d), "first": d.min(), "last": d.max()})
sp = pd.DataFrame(rows)
tab = sp.groupby(["family", "block"]).agg(dates=("dates", "sum"), first=("first", "min"), last=("last", "max")).unstack("block")
display(tab.reindex(columns=pd.MultiIndex.from_product([["dates", "first", "last"], [*BLOCKS3, "purged"]])))
print("one unit, t2-F3-inversion-persistence-2019 (horizons 63 and 126):")
display(sp[sp.unit == "t2-F3-inversion-persistence-2019"].set_index("block")[["dates", "first", "last"]])

dates                               first                                        last                                 
        train validation   test purged      train validation       test     purged      train validation       test     purged
family                                                                                                                        
T2-F1   47135      13308  16938   7316 2000-01-01 2002-08-16 2002-04-17 2000-12-13 2013-11-04 2018-08-15 2024-03-18 2019-05-15
T2-F2   58924      18410  20253   3684 1993-05-24 1999-03-31 2001-03-12 1998-12-28 2014-05-29 2019-04-18 2024-06-06 2019-07-19
T2-F3   59295      18463  20417   3905 1995-01-03 1999-09-29 2001-04-25 1999-04-01 2014-06-23 2019-05-29 2024-08-01 2019-08-27
T2-F4   68287      22176  23053   1744 2000-01-01 2004-06-28 2005-12-22 2004-05-27 2014-07-29 2019-06-28 2024-07-01 2019-07-30

one unit, t2-F3-inversion-persistence-2019 (horizons 63 and 126):


,dates,first,last
block,,,
train,2743,2000-01-03,2010-12-17
validation,831,2011-06-21,2014-10-15
test,956,2015-04-20,2019-02-13
purged,252,2010-12-20,2015-04-17


## 2 — The model with its four hyperparameters

`walk_forecast(...)` forecasts one date, using only the steps before it:

1. the last `window` steps give the correlation between assets → Cholesky factor `L`;
2. the sd of one step: an EWMA over the last `8 × halflife` steps (a step's weight halves every `halflife`), or the plain
   sd of the `window` steps when `halflife` is off; `scale = sd × widen`;
3. one path per draw through the horizons: add `L·e × t × scale × √(new steps)`, with `t = √((ν−2)/W)`,
   `W ~ χ²(ν)` for Student-t shocks (`t = 1` when normal).

With one asset the correlation is trivially 1, so `window` only matters when EWMA is off. The check below: at
production's current settings (`forecast_models.WALK_SETTINGS`) this is exactly production's model.

In [3]:
MIN_CHANGES = 10     # fewer steps than this before a date: the date is skipped

def ewma_sd(past, halflife):
    age = np.arange(len(past))[::-1]                  # newest step = age 0
    w = 0.5 ** (age / halflife); w = w / w.sum()
    mean = w @ past
    return np.sqrt(w @ (past - mean) ** 2)

def walk_forecast(steps, origin, anchor, ksteps, horizons, window=260, widen=1.0, nu=None, halflife=None,
                  n_draws=500, seed=0):
    """-> draws (n_draws, n_assets, n_horizons)"""
    i = steps.index.searchsorted(pd.Timestamp(origin), side="right")
    arr = steps.to_numpy()
    past = arr[max(0, i - window): i]
    if len(past) < MIN_CHANGES:
        raise ValueError(f"only {len(past)} steps before {origin}")
    n_a = past.shape[1]
    D = np.ascontiguousarray(past.T)
    sd = (D.std(axis=1) if halflife is None else ewma_sd(arr[max(0, i - int(8 * halflife)): i], halflife)) * widen
    with np.errstate(invalid="ignore", divide="ignore"):
        corr = np.nan_to_num(np.atleast_2d(np.corrcoef(D)), nan=0.0)
    np.fill_diagonal(corr, 1.0)
    w, v = np.linalg.eigh(corr)
    L = np.linalg.cholesky(v @ np.diag(np.clip(w, 1e-8, None)) @ v.T)
    rng, rng_t = np.random.default_rng(seed), np.random.default_rng([seed, 7])
    out = np.empty((n_draws, n_a, len(horizons)))
    path, prev = np.zeros((n_draws, n_a)), np.zeros(n_a)
    for hi in np.argsort(horizons):
        z = rng.standard_normal((n_draws, n_a)) @ L.T
        rng.standard_normal((n_draws, n_a))                        # production's skew draw (unused here)
        if nu is not None:
            z = z * np.sqrt((nu - 2.0) / rng_t.chisquare(nu, size=(n_draws, 1)))
        kk = np.asarray(ksteps, float)[:, hi]
        path = path + z * sd * np.sqrt(np.maximum(kk - prev, 0))
        prev = kk
        out[:, :, hi] = np.asarray(anchor, float) + path
    return out

def card_inputs(u):
    card = CARDS[u]; t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    steps = STEPS[STEPS.unit == u].pivot(index="date", columns="asset", values="step")[assets].sort_index()
    tt = TARGETS[TARGETS.unit == u].set_index(["origin_date", "asset", "horizon"]).sort_index()
    return card, assets, horizons, steps, tt

def at_origin(tt, origin, assets, horizons):
    r = tt.loc[origin]
    anchor = np.array([r.loc[(a, horizons[0]), "anchor"] for a in assets])
    ksteps = np.array([[r.loc[(a, h), "steps"] for h in horizons] for a in assets])
    y = np.array([r.loc[(a, h), "target"] for a in assets for h in horizons])
    return anchor, ksteps, y

import forecast_agent as fa, forecast_models as fm
rows = []
for u in ["t2-F1-pause-2006", "t2-F2-whatever-it-takes-2012", "t2-F3-boj-ust-channel-2023", "t2-F4-short-vol-2018"]:
    card, assets, horizons, steps, tt = card_inputs(u); t = card["targets"]
    o = SPLIT[u][SPLIT[u] == "test"].index[50]
    anchor, ksteps, y = at_origin(tt, o, assets, horizons)
    cfg = fm.WALK_SETTINGS[FAMILY[u]]
    mine = walk_forecast(steps, o, anchor, ksteps, horizons, n_draws=1000, seed=3, **cfg)
    prod = fm.build_draws(fa._read_panels(UNITS / u), assets, horizons, str(o.date()), {a: {} for a in assets}, 1000, 3,
                          target_type=t["target_type"], family=FAMILY[u])
    rows.append({"unit": u, "date": o.date(), "production setting": cfg, "max |mine - production|": float(np.abs(mine - prod).max())})
display(pd.DataFrame(rows))

,unit,date,production setting,max |mine - production|
0,t2-F1-pause-2006,2004-11-18,"{'window': 42, 'widen': 1.2, 'nu': 5, 'halflif...",0
1,t2-F2-whatever-it-takes-2012,2010-01-21,"{'window': 130, 'widen': 1.1, 'nu': 4, 'halfli...",0
2,t2-F3-boj-ust-channel-2023,2018-10-25,"{'window': 130, 'widen': 1.2, 'nu': 4, 'halfli...",0
3,t2-F4-short-vol-2018,2014-08-06,"{'window': 260, 'widen': 1.2, 'nu': 5, 'halfli...",0


## 3 — The score: each unit's total

`total_score` = the unit's `[scoring.params]` through the official composite function. Checked against the official
`scoring/scoring.py` on one unit.

In [4]:
from qfbench2_track_forecasting import scoring as S

def total_score(card, samples, y):
    """samples (n_draws, n_cells) in card order; y (n_cells,). Lower is better."""
    p = card.get("scoring", {}).get("params", {})
    w = p.get("weights", {"marginal": 0.5, "joint": 0.3, "tail": 0.2})
    w = (float(w["marginal"]), float(w["joint"]), float(w["tail"]))
    if samples.shape[1] == 1:
        live = w[0] + w[2]; w = (w[0] / live, 0.0, w[2] / live)
    return float(S._composite(samples, y, weights=w, tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))),
                              joint=S.card_joint_statistic(card), tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)),
                              ref_scale=None)["composite"])

u = "t2-F3-boj-ust-channel-2023"
card, assets, horizons, steps, tt = card_inputs(u); t = card["targets"]
o = SPLIT[u][SPLIT[u] == "test"].index[100]
anchor, ksteps, y = at_origin(tt, o, assets, horizons)
d3 = walk_forecast(steps, o, anchor, ksteps, horizons, window=130, widen=1.1, nu=5, halflife=42)
with tempfile.TemporaryDirectory() as tmp:
    out_dir, real = pathlib.Path(tmp) / "out", pathlib.Path(tmp) / "realized.parquet"; out_dir.mkdir()
    pd.DataFrame([{"draw": np.int32(d), "asset": a, "horizon": np.int32(h), "value": float(d3[d, ai, hi])}
                  for d in range(len(d3)) for ai, a in enumerate(assets) for hi, h in enumerate(horizons)]).to_parquet(out_dir / "forecast.parquet", index=False)
    (out_dir / "forecast_meta.json").write_text(json.dumps({"unit_id": card["task"]["id"], "asof": card["provenance"]["data_cutoff"],
        "representation": "samples", "asset_ids": assets, "horizons": horizons, "n_draws": len(d3), "target": t["target_type"]}))
    (out_dir / "forecast_rationale.md").write_text("# check\n")
    pd.DataFrame([{"asset": a, "horizon": h, "value": float(v)} for (a, h), v in zip([(a, h) for a in assets for h in horizons], y)]).to_parquet(real, index=False)
    r = subprocess.run([sys.executable, str(REPO / "scoring" / "scoring.py"), "score", "--card", str(UNITS / u / "card.toml"),
                        "--forecast", str(out_dir / "forecast.parquet"), "--realized", str(real)], capture_output=True, text=True, cwd=REPO)
official, mine = json.loads(r.stdout)["composite_score"], total_score(card, d3.reshape(len(d3), -1), y)
print(f"total_score {mine:.10f}   official scorer {official:.10f}   equal: {np.isclose(mine, official, rtol=0, atol=1e-12)}")
assert np.isclose(mine, official, rtol=0, atol=1e-12)

total_score 0.6564880971   official scorer 0.6564880971   equal: True


## 4 — The grid and the runner

`score_block(unit, block, settings)` forecasts every 5th date of the block with each setting, 500 draws, and returns
each setting's **mean total score** over those dates. Every setting at a date uses the **same random numbers** (the seed
is the date's position), so a difference between settings is the settings, not luck. With one asset, settings that
differ only in `window` (EWMA on) are the same forecast, so each is computed once. Units run in parallel.

In [5]:
WINDOWS = [42, 130, 260, 520]
HALFLIVES = [21, 42, 63, 126, 252, None]
NUS = [4, 5, 8, None]
WIDENS = [0.9, 1.0, 1.1, 1.2, 1.3]
GRID = [{"window": w, "widen": wd, "nu": nu, "halflife": hl} for w in WINDOWS for hl in HALFLIVES for nu in NUS for wd in WIDENS]
STRIDE, N_DRAWS = 5, 500

def key(s):
    return (f'window {int(s["window"])} / widen {float(s["widen"]):.1f} / '
            f'{"normal" if s["nu"] is None else "nu " + str(int(s["nu"]))} / '
            f'{"EWMA off" if s["halflife"] is None else "halflife " + str(int(s["halflife"]))}')

def setting_of(k):
    w, wd, nu, hl = [x.strip() for x in k.split("/")]
    return {"window": int(w.split()[1]), "widen": float(wd.split()[1]), "nu": None if nu == "normal" else int(nu.split()[1]),
            "halflife": None if hl == "EWMA off" else int(hl.split()[1])}

def effective(s, n_assets):
    """One asset + EWMA on: window has no effect, so these settings share one forecast."""
    return {**s, "window": 260} if (n_assets == 1 and s["halflife"] is not None) else s

def score_block(args):
    u, block, settings = args
    card, assets, horizons, steps, tt = card_inputs(u)
    eff = {key(s): key(effective(s, len(assets))) for s in settings}
    uniq = {k: setting_of(k) for k in set(eff.values())}
    total, n = dict.fromkeys(uniq, 0.0), 0
    for j, o in enumerate(SPLIT[u][SPLIT[u] == block].index[::STRIDE]):
        anchor, ksteps, y = at_origin(tt, o, assets, horizons)
        try:
            sc = {k: total_score(card, walk_forecast(steps, o, anchor, ksteps, horizons, n_draws=N_DRAWS, seed=j, **s)
                                 .reshape(N_DRAWS, -1), y) for k, s in uniq.items()}
        except ValueError:
            continue
        for k in uniq: total[k] += sc[k]
        n += 1
    return [{"unit": u, "family": FAMILY[u], "block": block, "setting": k, "score": total[e] / n if n else np.nan, "dates": n}
            for k, e in eff.items()]

def run(jobs):
    with mp.get_context("fork").Pool(WORKERS) as pool:
        out = pool.map(score_block, jobs, chunksize=1)
    return pd.DataFrame([r for rows in out for r in rows])

print(f"grid: {len(GRID)} settings = {len(WINDOWS)} windows x {len(HALFLIVES)} halflives x {len(NUS)} nu x {len(WIDENS)} widen")

grid: 480 settings = 4 windows x 6 halflives x 4 nu x 5 widen


## 5 — All 480 settings on validation; pick the best per unit and the global per family

- **best per unit**: the lowest validation score on that unit;
- **global per family**: the setting with the lowest **geometric mean** of validation scores over the family's units.
  (Units have different scales; the geometric mean weighs each unit equally.)

In [6]:
t0 = time.time()
VAL = run([(u, "validation", GRID) for u in CARDS])
print(f"validation grid: {VAL.drop_duplicates('unit').dates.sum():,} dates x {len(GRID)} settings, {time.time() - t0:.0f}s")
ok = VAL.dropna(subset=["score"])
BEST_UNIT = ok.loc[ok.groupby("unit").score.idxmin()].set_index("unit").setting.to_dict()
fam_geo = ok.groupby(["family", "setting"]).score.apply(geo)
GLOBAL = {f: fam_geo.loc[f].idxmin() for f in FAMS}
print("\nglobal per family (picked on validation):")
for f in FAMS: print(f"  {f}: {GLOBAL[f]}")
print("\nthe 5 best settings per family (validation, geometric mean ÷ the family's best):")
for f in FAMS:
    top = (fam_geo.loc[f] / fam_geo.loc[f].min()).sort_values().head(5).rename("÷ best")
    print(f"\n{f}"); display(top.to_frame())

validation grid: 14,518 dates x 480 settings, 157s

global per family (picked on validation):
  T2-F1: window 130 / widen 0.9 / nu 5 / halflife 21
  T2-F2: window 520 / widen 0.9 / nu 4 / EWMA off
  T2-F3: window 260 / widen 1.0 / nu 4 / halflife 126
  T2-F4: window 42 / widen 0.9 / normal / halflife 63

the 5 best settings per family (validation, geometric mean ÷ the family's best):

T2-F1


,÷ best
setting,
window 130 / widen 0.9 / nu 5 / halflife 21,1
window 260 / widen 0.9 / nu 5 / halflife 21,1
window 42 / widen 0.9 / nu 5 / halflife 21,1
window 520 / widen 0.9 / nu 5 / halflife 21,1
window 130 / widen 0.9 / nu 5 / halflife 42,1



T2-F2


,÷ best
setting,
window 520 / widen 0.9 / nu 4 / EWMA off,1
window 520 / widen 0.9 / nu 4 / halflife 63,1.001
window 130 / widen 0.9 / nu 4 / halflife 63,1.001
window 260 / widen 0.9 / nu 4 / halflife 63,1.001
window 42 / widen 0.9 / nu 4 / halflife 63,1.001



T2-F3


,÷ best
setting,
window 260 / widen 1.0 / nu 4 / halflife 126,1
window 130 / widen 1.0 / nu 4 / halflife 126,1
window 260 / widen 1.0 / nu 5 / halflife 126,1
window 130 / widen 1.0 / nu 5 / halflife 126,1
window 520 / widen 1.0 / nu 4 / halflife 126,1.001



T2-F4


,÷ best
setting,
window 42 / widen 0.9 / normal / halflife 63,1
window 520 / widen 0.9 / normal / halflife 63,1
window 42 / widen 0.9 / normal / halflife 126,1
window 520 / widen 0.9 / normal / halflife 126,1.001
window 42 / widen 0.9 / nu 8 / halflife 126,1.001


In [7]:
print("How often each value is the best for a unit (per family):")
bu = pd.DataFrame([{**setting_of(s), "unit": u, "family": FAMILY[u]} for u, s in BEST_UNIT.items()])
bu["window"] = [("any (1 asset)" if (NASSETS[u] == 1 and pd.notna(h)) else str(int(w))) for u, w, h in zip(bu.unit, bu.window, bu.halflife)]
print(f"{len(bu)} units have a validation score; {len(CARDS) - len(bu)} have no validation date (too little history) "
      f"and take their family's global: {sorted(set(CARDS) - set(bu.unit))}")
FMT = {"window": str, "halflife": lambda v: "off" if pd.isna(v) else str(int(v)),
       "nu": lambda v: "normal" if pd.isna(v) else str(int(v)), "widen": lambda v: f"{v:.1f}"}
for col in ["window", "halflife", "nu", "widen"]:
    c = bu.assign(v=bu[col].map(FMT[col]))
    print(f"\n{col}:"); display(c.groupby(["family", "v"]).size().unstack(fill_value=0))

How often each value is the best for a unit (per family):
101 units have a validation score; 2 have no validation date (too little history) and take their family's global: ['t2-F1-considerable-period-2003', 't2-F1-measured-pace-2004']

window:


v,130,260,42,520,any (1 asset)
family,,,,,
T2-F1,2,1,6,6,6
T2-F2,4,2,1,8,12
T2-F3,8,4,5,5,0
T2-F4,1,8,7,5,10



halflife:


v,126,21,252,63,off
family,,,,,
T2-F1,0,3,3,0,15
T2-F2,0,5,7,0,15
T2-F3,2,2,6,2,10
T2-F4,2,4,4,1,20



nu:


v,4,5,8,normal
family,,,,
T2-F1,7,3,2,9
T2-F2,11,6,3,7
T2-F3,8,4,7,3
T2-F4,3,5,10,13



widen:


v,0.9,1.0,1.1,1.2,1.3
family,,,,,
T2-F1,15,0,2,2,2
T2-F2,15,2,4,2,4
T2-F3,11,6,3,0,2
T2-F4,19,6,2,2,2


## 6 — Score the picks on train and test; the per-unit report

Each unit's own best, its family's global, and — for reference, since production may change later — production's
current setting, on **train**, **validation** and **test** (every 5th date, 500 draws, same random numbers). The
validation column of a pick is where it was picked, so it is optimistic; **test is the honest score.**

In [8]:
PROD = {f: key(fm.WALK_SETTINGS[f]) for f in FAMS}
assert all(PROD[f] in {key(g) for g in GRID} for f in FAMS)
picks = {u: {BEST_UNIT.get(u, GLOBAL[FAMILY[u]]), GLOBAL[FAMILY[u]], PROD[FAMILY[u]]} for u in CARDS}
t0 = time.time()
TT = run([(u, b, [setting_of(k) for k in picks[u]]) for u in CARDS for b in ("train", "test")])
print(f"train + test: {time.time() - t0:.0f}s")
ALL = pd.concat([VAL, TT], ignore_index=True)
score = ALL.set_index(["unit", "block", "setting"]).score

def three(u, k):
    return [score.get((u, b, k), np.nan) for b in BLOCKS3]

rows = []
for u in CARDS:
    f, b = FAMILY[u], BEST_UNIT.get(u, GLOBAL[FAMILY[u]])
    s = setting_of(b)
    rows.append({"family": f, "unit": u, "assets": NASSETS[u],
                 "window": "any" if (NASSETS[u] == 1 and s["halflife"] is not None) else s["window"],
                 "halflife": s["halflife"] or "off", "nu": s["nu"] or "normal", "widen": s["widen"],
                 **dict(zip(["best: train", "best: validation", "best: test"], three(u, b))),
                 **dict(zip(["global: train", "global: validation", "global: test"], three(u, GLOBAL[f])))})
PER_UNIT = pd.DataFrame(rows).set_index(["family", "unit"])
print("Best hyperparameters per unit (picked on validation), and the total score of that pick and of the family's global:")
PER_UNIT

train + test: 10s
Best hyperparameters per unit (picked on validation), and the total score of that pick and of the family's global:


assets window halflife      nu  widen  best: train  best: validation  best: test  global: train  global: validation  global: test
family unit                                                                                                                                                                
T2-F1  t2-F1-ai-mom-2024                       1    130      off  normal    1.1      0.05206            0.0356     0.04552        0.05252             0.03714       0.04333
       t2-F1-aud-on-hold-2016                  1    520      off       4    0.9      0.03107           0.02505      0.0296        0.02992             0.02612       0.03217
       t2-F1-cad-boc-2017                      1     42      off       4    0.9      0.03622           0.01439     0.03553        0.03564             0.01476       0.03513
       t2-F1-chf-highly-valued-2021            1    any      252       4    0.9      0.03556           0.01691     0.01335        0.03636             0.01799        0.0125
       t2-F1-conflicting-texts-2024            1    260      off  normal    0.9       0.2399            0.1684      0.3438         0.2416              0.1748        0.3464
       t2-F1-considerable-period-2003          1    any       21       5    0.9       0.2925               NaN       0.282         0.2925                 NaN         0.282
       t2-F1-conundrum-2005                    1    any       21       5    0.9       0.2309            0.1867      0.1601         0.2309              0.1867        0.1601
       t2-F1-cpi-glidepath-2023                1    130      off  normal    1.3        1.621             1.069       3.931          1.781               1.258         4.186
       t2-F1-dkk-peg-2019                      1     42      off  normal    0.9        0.222            0.1198      0.1534         0.2233              0.1207         0.153
       t2-F1-eur-range-2017                    1    any       21  normal    0.9       0.0388           0.03016     0.03776        0.03945             0.03039       0.03779
       t2-F1-fed-put-2019                      1    520      off  normal    0.9      0.06053            0.0434     0.03423        0.06009             0.04578       0.03673
       t2-F1-greater-confidence-2024           1    520      off       8    1.2       0.3267            0.1365      0.5198         0.3142               0.151        0.5415
       t2-F1-hawkish-cut-2024                  1    520      off       8    1.3       0.3193            0.1511      0.4819         0.3039              0.1711        0.4981
       t2-F1-jpy-ycc-flex-2018                 1    any      252       5    1.1        2.776             3.069        3.54          2.841               3.306         3.761
       t2-F1-liftoff-telegraph-2015            1     42      off       4    0.9       0.4226            0.1214     0.04539         0.4164               0.123       0.04598
       t2-F1-measured-pace-2004                1    any       21       5    0.9       0.6867               NaN      0.1801         0.6867                 NaN        0.1801
       t2-F1-normalization-2017                1    520      off  normal    0.9       0.2616            0.2976      0.1858         0.2617              0.3151        0.1913
       t2-F1-patient-dropped-2015              1     42      off       4    0.9       0.4397            0.1428     0.04237         0.4337              0.1457       0.04338
       t2-F1-pause-2006                        2     42      off       4    1.2       0.6797            0.3509      0.7205          0.732              0.3657        0.7231
       t2-F1-qe3-end-2014                      1    any      252  normal    0.9       0.2151            0.2676      0.2193          0.215              0.2851        0.2232
       t2-F1-sahm-watch-2024                   1    any       21  normal    0.9       0.3131             0.125       1.179         0.3153              0.1263         1.124
       t2-F1-third-cut-2019                    1     42      off       4    0.9       

## 7 — Per family

The family's global setting, and the family's total score (mean and median over its units, since the mean is led by the
units with large scores) on train, validation and test, for three choices: **each unit's own best**, **the family
global**, and **production's current setting**. `test ÷ global` = geometric mean over the family's units of (that
choice's test score ÷ the global's test score): below 1, the choice beats the global on test.

In [9]:
CHOICES = {"best per unit": lambda u: BEST_UNIT.get(u, GLOBAL[FAMILY[u]]), "global per family": lambda u: GLOBAL[FAMILY[u]],
           "production now": lambda u: PROD[FAMILY[u]]}
rows = []
for f in [*FAMS, "all"]:
    us = [u for u in CARDS if f == "all" or FAMILY[u] == f]
    for name, pick in CHOICES.items():
        sc = pd.DataFrame([three(u, pick(u)) for u in us], index=us, columns=BLOCKS3)
        g = pd.Series([score.get((u, "test", GLOBAL[FAMILY[u]]), np.nan) for u in us], index=us)
        ratio = (sc.test / g).dropna()
        label = ("per unit" if name == "best per unit" else "per family" if f == "all"
                 else GLOBAL[f] if name == "global per family" else PROD[f])
        rows.append({"family": f, "choice": name, "setting": label,
                     **{f"{b} mean": sc[b].mean() for b in BLOCKS3}, **{f"{b} median": sc[b].median() for b in BLOCKS3},
                     "test ÷ global": geo(ratio), "better than global on test": f"{(ratio < 1).sum()}/{len(ratio)}"})
PER_FAMILY = pd.DataFrame(rows).set_index(["family", "choice"])
PER_FAMILY

setting  train mean  validation mean  test mean  train median  validation median  test median  test ÷ global better than global on test
family choice                                                                                                                                                                                         
T2-F1  best per unit                                          per unit      0.4308           0.3177     0.5467        0.2616             0.1365       0.1801         0.9882                      14/23
       global per family   window 130 / widen 0.9 / nu 5 / halflife 21      0.4411           0.3436     0.5674        0.2617             0.1457       0.1801              1                       0/23
       production now          window 42 / widen 1.2 / nu 5 / EWMA off      0.4276           0.3393     0.5686        0.2654             0.1501       0.1871          1.027                      11/23
T2-F2  best per unit                                          per unit      0.4583           0.4204     0.4435        0.1999             0.1057       0.1938          1.017                       9/27
       global per family      window 520 / widen 0.9 / nu 4 / EWMA off      0.4613           0.4362     0.4383        0.1999             0.1096       0.1799              1                       0/27
       production now         window 130 / widen 1.1 / nu 4 / EWMA off      0.4551           0.4288     0.4367        0.1905             0.1105       0.1844         0.9879                      13/27
T2-F3  best per unit                                          per unit      0.8951           0.7245      0.798         1.059             0.5352       0.7639          1.021                      10/22
       global per family  window 260 / widen 1.0 / nu 4 / halflife 126       0.889           0.7424     0.7763         1.026             0.5438       0.7234              1                       0/22
       production now         window 130 / widen 1.2 / nu 4 / EWMA off      0.8831           0.7555     0.7762         1.036             0.5577       0.7318          1.002                      12/22
T2-F4  best per unit                                          per unit       6.124            6.965      5.776       0.03996            0.02539      0.04247          1.006                      10/31
       global per family  window 42 / widen 0.9 / normal / halflife 63       6.365            7.564      6.216       0.03917            0.02572      0.04399              1                       0/31
       production now      window 260 / widen 1.2 / nu 5 / halflife 63       6.185            7.407      6.058       0.03886             0.0257       0.0427         0.9947                      19/31
all    best per unit                                          per unit       2.251            2.474      2.147        0.2151             0.1214       0.1858          1.008                     43/103
       global per family                                    per family       2.325            2.671      2.278         0.215              0.123       0.1801              1                      0/103
       production now                                       per family       2.265            2.623      2.231        0.2161             0.1271       0.1871          1.002                     55/103

## 8 — Save

In [10]:
PER_UNIT.to_csv(OUT / "best_params_per_unit.csv")
pd.DataFrame({f: {"setting": GLOBAL[f], **setting_of(GLOBAL[f])} for f in FAMS}).T.rename_axis("family").to_csv(OUT / "global_params_per_family.csv")
PER_FAMILY.to_csv(OUT / "scores_per_family.csv")
VAL.to_csv(OUT / "validation_grid.csv.gz", index=False)
for f in ("best_params_per_unit.csv", "global_params_per_family.csv", "scores_per_family.csv", "validation_grid.csv.gz"):
    print(f"data/{f:<30} {(OUT / f).stat().st_size / 1e3:8.1f} kB")

data/best_params_per_unit.csv           17.2 kB
data/global_params_per_family.csv        0.3 kB
data/scores_per_family.csv               2.9 kB
data/validation_grid.csv.gz            466.8 kB


## 9 — In short

In [11]:
pf = PER_FAMILY
for f in FAMS:
    g = pf.loc[(f, "global per family")]; b = pf.loc[(f, "best per unit")]; p = pf.loc[(f, "production now")]
    print(f"{f}: global = {GLOBAL[f]}")
    print(f"      test median score: best per unit {b['test median']:.4g} | global {g['test median']:.4g} | production now {p['test median']:.4g}")
    print(f"      test ÷ global: best per unit {b['test ÷ global']:.3f} ({b['better than global on test']} units better),"
          f" production now {p['test ÷ global']:.3f}")
a = pf.loc[("all", "best per unit")]; pa = pf.loc[("all", "production now")]
print(f"\nall units: best per unit ÷ global on test = {a['test ÷ global']:.3f}; production now ÷ global = {pa['test ÷ global']:.3f}")
print("-> " + ("per-unit settings carry over to test better than one global per family."
               if a["test ÷ global"] < 0.99 else
               "per-unit settings do not beat one global per family on test: use the family global."
               if a["test ÷ global"] >= 1.0 else "per-unit and family-global settings are within 1% on test."))

T2-F1: global = window 130 / widen 0.9 / nu 5 / halflife 21
      test median score: best per unit 0.1801 | global 0.1801 | production now 0.1871
      test ÷ global: best per unit 0.988 (14/23 units better), production now 1.027
T2-F2: global = window 520 / widen 0.9 / nu 4 / EWMA off
      test median score: best per unit 0.1938 | global 0.1799 | production now 0.1844
      test ÷ global: best per unit 1.017 (9/27 units better), production now 0.988
T2-F3: global = window 260 / widen 1.0 / nu 4 / halflife 126
      test median score: best per unit 0.7639 | global 0.7234 | production now 0.7318
      test ÷ global: best per unit 1.021 (10/22 units better), production now 1.002
T2-F4: global = window 42 / widen 0.9 / normal / halflife 63
      test median score: best per unit 0.04247 | global 0.04399 | production now 0.0427
      test ÷ global: best per unit 1.006 (10/31 units better), production now 0.995

all units: best per unit ÷ global on test = 1.008; production now ÷ global = 1.